### Data Generation

Realistics synthetic data of 50K rows.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
np.random.seed(42)

N = 50_000
print(f"Generating {N:,} customer records...")

Generating 50,000 customer records...


In [3]:
# --- Customer base ---
customer_ids = np.arange(1, N + 1)
signup_dates = pd.date_range("2023-01-01", periods=N, freq="1h")
plan_type = np.random.choice(["free", "basic", "pro"], N, p=[0.50, 0.30, 0.20])

# --- Behavioral features (correlated with churn) ---
# Pro users have longer sessions; free users shorter
base_session = np.where(plan_type == "pro", 300,
               np.where(plan_type == "basic", 180, 90))
session_duration = np.random.exponential(base_session).clip(0, 3600).astype("int64")

# Pages viewed — higher for engaged users
pages_viewed = np.random.poisson(
    np.where(plan_type == "pro", 8, np.where(plan_type == "basic", 5, 3))
).clip(0, 50)

# Support tickets — churners raise more tickets
support_tickets = np.random.poisson(
    np.where(plan_type == "free", 0.8, 0.2)
).clip(0, 10)

# Days inactive — churners go quiet before leaving
days_inactive = np.random.exponential(
    np.where(plan_type == "pro", 5, np.where(plan_type == "basic", 10, 20))
).clip(0, 90).astype(int)

# Login count last 30 days
login_count_30d = np.random.poisson(
    np.where(plan_type == "pro", 20, np.where(plan_type == "basic", 10, 4))
).clip(0, 60)

# Feature used count (product engagement)
features_used = np.random.poisson(
    np.where(plan_type == "pro", 8, np.where(plan_type == "basic", 4, 1))
).clip(0, 20)

# --- Churn label (realistically correlated with features) ---
# Build churn probability from feature signals
churn_score = (
    0.3 * (days_inactive / 90)           # longer inactive → higher churn
    + 0.2 * (support_tickets / 10)        # more tickets → higher churn
    - 0.2 * (session_duration / 3600)     # longer sessions → lower churn
    - 0.15 * (login_count_30d / 60)       # more logins → lower churn
    - 0.1 * (features_used / 20)          # more features used → lower churn
    + 0.05 * (plan_type == "free").astype(float)
    + np.random.normal(0, 0.05, N)        # noise
).clip(0, 1)

churned = (churn_score > np.percentile(churn_score, 93)).astype(int)  # ~7% churn rate

# --- Assemble DataFrame ---
df = pd.DataFrame({
    "customer_id":      customer_ids,
    "signup_date":      signup_dates,
    "plan_type":        plan_type,
    "session_duration_sec": session_duration,
    "pages_viewed":     pages_viewed,
    "support_tickets":  support_tickets,
    "days_inactive":    days_inactive,
    "login_count_30d":  login_count_30d,
    "features_used":    features_used,
    "churned":          churned,
})

# --- Introduce realistic data issues ---
# 3% null rate on session_duration (mimics tracking failures)
null_mask = np.random.rand(N) < 0.03
df.loc[null_mask, "session_duration_sec"] = np.nan

# A few outliers (data entry errors)
df.loc[np.random.choice(N, 10), "session_duration_sec"] = 99999

Path("data").mkdir(exist_ok=True)
df.to_parquet("data/raw_events.parquet", index=False)

print(f"Saved: data/raw_events.parquet")
print(f"\nDataset shape: {df.shape}")
print(f"Churn rate: {churned.mean()*100:.1f}%")
print(f"Null rate (session_duration): {null_mask.mean()*100:.1f}%")
print(f"\nSample:\n{df.head(3).to_string()}")

Saved: data/raw_events.parquet

Dataset shape: (50000, 10)
Churn rate: 7.0%
Null rate (session_duration): 2.9%

Sample:
   customer_id         signup_date plan_type  session_duration_sec  pages_viewed  support_tickets  days_inactive  login_count_30d  features_used  churned
0            1 2023-01-01 00:00:00      free                 169.0             4                0              2                4              0        0
1            2 2023-01-01 01:00:00       pro                 204.0             7                0              3               18             12        0
2            3 2023-01-01 02:00:00     basic                  39.0             5                0              4                6              6        0
